<a href="https://colab.research.google.com/github/MostachoteRex/analizador_rendimiento/blob/main/analizador_rendimiento.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Paso 1 - Configuración Inicial

In [ ]:
import time
import statistics
import functools
import datetime
from collections import defaultdict
import math

print("Módulos importados correctamente:")
print(f"   - time: {time.__name__}")
print(f"   - statistics: {statistics.__name__}")
print(f"   - functools: {functools.__name__}")
print(f"   - datetime: {datetime.__name__}")
print(f"   - collections.defaultdict: Disponible")
print(f"   - math: {math.__name__}")

# Verificación de la existencia del archivo Logs_ejemplo.txt
import os

if os.path.exists('Logs_ejemplo.txt'):
    print("Archivo Logs_ejemplo.txt encontrado")
else:
    print("Archivo Logs_ejemplo.txt NO encontrado")
    print("Por favor, sube el archivo manualmente")

print("\nConfiguración lista para el análisis de rendimiento")

Módulos importados correctamente:
   - time: time
   - statistics: statistics
   - functools: functools
   - datetime: datetime
   - collections.defaultdict: Disponible
   - math: math
Archivo Logs_ejemplo.txt encontrado

Configuración lista para el análisis de rendimiento


Paso 2 - Crear Funciones Base

In [ ]:
def parsear_log(linea):
    try:
        partes = linea.strip().split()

        # Verificar que tenga al menos 5 componentes
        if len(partes) < 5:
            return None

        # Extraer componentes
        fecha = partes[0]
        hora = partes[1]
        metodo = partes[2]
        endpoint = partes[3]
        codigo_estado = int(partes[4])
        tiempo_respuesta = float(partes[5])

        # Crear timestamp combinado
        timestamp_str = f"{fecha} {hora}"

        return {
            'timestamp': timestamp_str,
            'endpoint': endpoint,
            'metodo': metodo,
            'codigo_estado': codigo_estado,
            'tiempo_respuesta': tiempo_respuesta
        }

    except (ValueError, IndexError) as e:
        print(f"Error parseando línea: {linea[:50]}... - {e}")
        return None

def calcular_percentiles(tiempos, percentiles=[50, 90, 95, 99]):
    if not tiempos:
        return {f'p{p}': 0 for p in percentiles}

    # Ordenar los tiempos
    tiempos_ordenados = sorted(tiempos)
    n = len(tiempos_ordenados)

    resultados = {}

    for p in percentiles:
        # Calcular índice para el percentil
        indice = (p / 100) * (n - 1)

        if indice.is_integer():
            # Si el índice es entero, tomar el valor directamente
            valor = tiempos_ordenados[int(indice)]
        else:
            # Interpolación lineal entre valores adyacentes
            indice_inf = int(indice)
            indice_sup = indice_inf + 1
            peso_sup = indice - indice_inf
            peso_inf = 1 - peso_sup

            valor = (tiempos_ordenados[indice_inf] * peso_inf +
                    tiempos_ordenados[indice_sup] * peso_sup)

        resultados[f'p{p}'] = round(valor, 4)

    return resultados

def clasificar_rendimiento(tiempo):
    clasificador = lambda t: (
        "crítica" if t > 2.0 else
        "lenta" if t > 1.0 else
        "normal" if t > 0.5 else
        "rápida"
    )

    return clasificador(tiempo)

if not os.path.exists('Logs_ejemplo.txt'):
    print("Archivo Logs_ejemplo.txt no encontrado")
else:
    logs_parseados = []
    tiempos_respuesta = []
    codigos_estado = []
    endpoints_unicos = set()

    with open('Logs_ejemplo.txt', 'r') as f:
        for num_linea, linea in enumerate(f, 1):
            log = parsear_log(linea)
            if log:
                logs_parseados.append(log)
                tiempos_respuesta.append(log['tiempo_respuesta'])
                codigos_estado.append(log['codigo_estado'])
                endpoints_unicos.add(log['endpoint'])

    print("\nESTADÍSTICAS CON DATOS")

    if tiempos_respuesta:
        percentiles = calcular_percentiles(tiempos_respuesta)
        clasificaciones = [clasificar_rendimiento(t) for t in tiempos_respuesta]

        conteo_clasificaciones = {
            categoria: clasificaciones.count(categoria)
            for categoria in set(clasificaciones)
        }

        conteo_codigos = {}
        for codigo in set(codigos_estado):
            conteo_codigos[codigo] = codigos_estado.count(codigo)

        print("\nENDPOINTS:")
        endpoints_contados = {}
        for log in logs_parseados:
            endpoint = log['endpoint']
            endpoints_contados[endpoint] = endpoints_contados.get(endpoint, 0) + 1
        top_endpoints = sorted(endpoints_contados.items(), key=lambda x: x[1], reverse=True)[:5]
        for endpoint, count in top_endpoints:
            print(f"   - {endpoint}: {count} solicitudes")

        print("\nTIEMPOS DE RESPUESTA:")
        print(f"   - Promedio: {statistics.mean(tiempos_respuesta):.3f}s")
        print(f"   - Máximo: {max(tiempos_respuesta):.3f}s")
        print(f"   - Mínimo: {min(tiempos_respuesta):.3f}s")
        print(f"   - Desviación estándar: {statistics.stdev(tiempos_respuesta):.3f}s")

        print("\nCÓDIGOS DE ESTADO:")
        for codigo, cantidad in sorted(conteo_codigos.items()):
            porcentaje = (cantidad / len(codigos_estado)) * 100
            print(f"   - {codigo}: {cantidad} solicitudes ({porcentaje:.1f}%)")

        print("\nPERCENTILES:")
        for percentil, valor in percentiles.items():
            print(f"   - {percentil}: {valor:.3f}s")

        print("\nCLASIFICACIÓN DE RENDIMIENTO:")
        for categoria, cantidad in sorted(conteo_clasificaciones.items()):
            porcentaje = (cantidad / len(tiempos_respuesta)) * 100
            print(f"   - {categoria.capitalize()}: {cantidad} solicitudes ({porcentaje:.1f}%)")

    # EJEMPLOS DE LOGS PARSEADOS
    print("\nEJEMPLOS DE LOGS PARSEADOS (primeros 5):")
    for i, log in enumerate(logs_parseados[:5]):
        print(f"   Ejemplo {i+1}:")
        print(f"     Timestamp: {log['timestamp']}")
        print(f"     Método: {log['metodo']}")
        print(f"     Endpoint: {log['endpoint']}")
        print(f"     Código: {log['codigo_estado']}")
        print(f"     Tiempo: {log['tiempo_respuesta']:.3f}s")
        print(f"     Clasificación: {clasificar_rendimiento(log['tiempo_respuesta'])}")
        print()


ESTADÍSTICAS CON DATOS

ENDPOINTS:
   - /api/users: 6 solicitudes
   - /api/products: 2 solicitudes
   - /api/login: 1 solicitudes
   - /api/orders: 1 solicitudes
   - /api/users/123: 1 solicitudes

TIEMPOS DE RESPUESTA:
   - Promedio: 0.255s
   - Máximo: 1.234s
   - Mínimo: 0.011s
   - Desviación estándar: 0.308s

CÓDIGOS DE ESTADO:
   - 200: 44 solicitudes (88.0%)
   - 201: 3 solicitudes (6.0%)
   - 404: 1 solicitudes (2.0%)
   - 413: 1 solicitudes (2.0%)
   - 500: 1 solicitudes (2.0%)

PERCENTILES:
   - p50: 0.123s
   - p90: 0.689s
   - p95: 0.891s
   - p99: 1.234s

CLASIFICACIÓN DE RENDIMIENTO:
   - Lenta: 2 solicitudes (4.0%)
   - Normal: 6 solicitudes (12.0%)
   - Rápida: 42 solicitudes (84.0%)

EJEMPLOS DE LOGS PARSEADOS (primeros 5):
   Ejemplo 1:
     Timestamp: 2024-10-17 10:23:45
     Método: GET
     Endpoint: /api/users
     Código: 200
     Tiempo: 0.045s
     Clasificación: rápida

   Ejemplo 2:
     Timestamp: 2024-10-17 10:23:46
     Método: POST
     Endpoint: /api/l

Paso 3 - Procesamiento con Lambdas

In [ ]:
logs_parseados = []
with open('Logs_ejemplo.txt', 'r') as f:
    for linea in f:
        log = parsear_log(linea)
        if log:
            logs_parseados.append(log)

print(f"{len(logs_parseados)} logs cargados")

print("\nFILTER() - Requests con código >= 400")
requests_con_error = list(filter(lambda log: log['codigo_estado'] >= 400, logs_parseados))

print(f"Encontrados: {len(requests_con_error)} requests con error")
for error_log in requests_con_error:
    print(f"  {error_log['codigo_estado']} - {error_log['metodo']} {error_log['endpoint']}")

print("\nMAP() - Timestamps formateados")

def formatear_timestamp(log):
    from datetime import datetime
    try:
        dt = datetime.strptime(log['timestamp'], '%Y-%m-%d %H:%M:%S')
        return dt.strftime('%d/%m/%Y %H:%M:%S')
    except:
        return log['timestamp']

timestamps_formateados = list(map(formatear_timestamp, logs_parseados[:5]))

print("Primeros 5 timestamps formateados:")
for i, timestamp in enumerate(timestamps_formateados):
    print(f"  {i+1}. {timestamp}")

print("\nSORTED() - Ordenado por tiempo_respuesta")

logs_ordenados = sorted(logs_parseados, key=lambda log: log['tiempo_respuesta'], reverse=True)

print("Top 5 más lentos:")
for i, log in enumerate(logs_ordenados[:5]):
    print(f"  {i+1}. {log['tiempo_respuesta']:.3f}s - {log['endpoint']}")

print("\nTop 5 más rápidos:")
for i, log in enumerate(logs_ordenados[-5:]):
    print(f"  {i+1}. {log['tiempo_respuesta']:.3f}s - {log['endpoint']}")

50 logs cargados

FILTER() - Requests con código >= 400
Encontrados: 3 requests con error
  500 - GET /api/orders
  404 - GET /api/users/456
  413 - POST /api/upload

MAP() - Timestamps formateados
Primeros 5 timestamps formateados:
  1. 17/10/2024 10:23:45
  2. 17/10/2024 10:23:46
  3. 17/10/2024 10:23:47
  4. 17/10/2024 10:23:48
  5. 17/10/2024 10:23:49

SORTED() - Ordenado por tiempo_respuesta
Top 5 más lentos:
  1. 1.234s - /api/upload
  2. 1.234s - /api/backup
  3. 0.892s - /api/orders
  4. 0.890s - /api/export
  5. 0.789s - /api/reports

Top 5 más rápidos:
  1. 0.012s - /api/cache
  2. 0.012s - /api/logout
  3. 0.012s - /api/health
  4. 0.011s - /api/temp
  5. 0.011s - /api/version


Paso 4 - Gestión de Estado

In [ ]:
print("DICCIONARIO GLOBAL UMBRALES")

UMBRALES = {'rapido': 0.5, 'normal': 1.0, 'lento': 2.0}

print("Umbrales definidos:")
for categoria, umbral in UMBRALES.items():
    print(f"  - {categoria}: {umbral}s")

print("\nCLOSURE contador_alertas()")

def crear_contador_alertas():
    """Closure que mantiene estado persistente del contador"""
    contador = 0

    def contador_alertas(tiempo_respuesta, incrementar=True):
        nonlocal contador

        if tiempo_respuesta > UMBRALES['lento']:
            categoria = "CRÍTICO"
            if incrementar:
                contador += 1
        elif tiempo_respuesta > UMBRALES['normal']:
            categoria = "LENTO"
            if incrementar:
                contador += 1
        else:
            categoria = "ACEPTABLE"

        return {
            'categoria': categoria,
            'contador_actual': contador,
            'tiempo_evaluado': tiempo_respuesta
        }

    return contador_alertas

contador = crear_contador_alertas()

print("Simulando alertas con closure:")
tiempos_test = [0.3, 1.5, 2.5, 0.8, 3.0]
for tiempo in tiempos_test:
    resultado = contador(tiempo)
    print(f"  {tiempo}s → {resultado['categoria']} (Alertas totales: {resultado['contador_actual']})")

print("\nDIFERENCIA: Modificar lista vs número")

def funcion_con_lista(lista_param, valor):
    """Modifica la lista original (mutable)"""
    lista_param.append(valor)
    return f"Lista modificada: {lista_param}"

def funcion_con_numero(numero_param, valor):
    """No modifica el número original (inmutable)"""
    numero_param += valor
    return f"Número modificado: {numero_param}"

lista_original = [1, 2, 3]
numero_original = 10

print("ANTES de llamar a las funciones:")
print(f"  lista_original: {lista_original}")
print(f"  numero_original: {numero_original}")

resultado_lista = funcion_con_lista(lista_original, 4)
resultado_numero = funcion_con_numero(numero_original, 5)

print("\nDESPUÉS de llamar a las funciones:")
print(f"  {resultado_lista}")
print(f"  {resultado_numero}")
print(f"  lista_original (MODIFICADA): {lista_original}")
print(f"  numero_original (NO modificado): {numero_original}")

print("\nEXPLICACIÓN:")
print("  - Listas son MUTABLES: se modifican en el lugar")
print("  - Números son INMUTABLES: se crea una copia local")

DICCIONARIO GLOBAL UMBRALES
Umbrales definidos:
  - rapido: 0.5s
  - normal: 1.0s
  - lento: 2.0s

CLOSURE contador_alertas()
Simulando alertas con closure:
  0.3s → ACEPTABLE (Alertas totales: 0)
  1.5s → LENTO (Alertas totales: 1)
  2.5s → CRÍTICO (Alertas totales: 2)
  0.8s → ACEPTABLE (Alertas totales: 2)
  3.0s → CRÍTICO (Alertas totales: 3)

DIFERENCIA: Modificar lista vs número
ANTES de llamar a las funciones:
  lista_original: [1, 2, 3]
  numero_original: 10

DESPUÉS de llamar a las funciones:
  Lista modificada: [1, 2, 3, 4]
  Número modificado: 15
  lista_original (MODIFICADA): [1, 2, 3, 4]
  numero_original (NO modificado): 10

EXPLICACIÓN:
  - Listas son MUTABLES: se modifican en el lugar
  - Números son INMUTABLES: se crea una copia local


Paso 5 - Integración Final

In [ ]:
def main():
    print("INICIANDO ANALISIS COMPLETO DE LOGS")

    logs_parseados = []
    with open('Logs_ejemplo.txt', 'r') as f:
        for linea in f:
            log = parsear_log(linea)
            if log:
                logs_parseados.append(log)

    print(f"{len(logs_parseados)} logs procesados correctamente")

    tiempos = [log['tiempo_respuesta'] for log in logs_parseados]
    codigos = [log['codigo_estado'] for log in logs_parseados]

    stats_general = {
        'total': len(logs_parseados),
        'promedio': statistics.mean(tiempos) * 1000,
        'maximo': max(tiempos) * 1000,
        'minimo': min(tiempos) * 1000,
        'desviacion': statistics.stdev(tiempos) * 1000 if len(tiempos) > 1 else 0
    }

    codigos_unicos = set(codigos)
    analisis_codigos = {}
    for codigo in codigos_unicos:
        logs_codigo = list(filter(lambda log: log['codigo_estado'] == codigo, logs_parseados))
        tiempos_codigo = [log['tiempo_respuesta'] for log in logs_codigo]
        analisis_codigos[codigo] = {
            'cantidad': len(logs_codigo),
            'porcentaje': (len(logs_codigo) / len(logs_parseados)) * 100,
            'tiempo_promedio': statistics.mean(tiempos_codigo) * 1000 if tiempos_codigo else 0
        }

    endpoints = {}
    for log in logs_parseados:
        endpoint = log['endpoint']
        if endpoint not in endpoints:
            endpoints[endpoint] = []
        endpoints[endpoint].append(log['tiempo_respuesta'])

    analisis_endpoints = {}
    for endpoint, tiempos_endpoint in endpoints.items():
        analisis_endpoints[endpoint] = {
            'solicitudes': len(tiempos_endpoint),
            'tiempo_promedio': statistics.mean(tiempos_endpoint) * 1000,
            'tiempo_maximo': max(tiempos_endpoint) * 1000,
            'tiempo_minimo': min(tiempos_endpoint) * 1000
        }

    endpoints_lentos = sorted(
        analisis_endpoints.items(),
        key=lambda x: x[1]['tiempo_promedio'],
        reverse=True
    )[:3]

    errores = list(filter(lambda log: log['codigo_estado'] >= 400, logs_parseados))

    generar_reporte_plantilla(stats_general, analisis_codigos, analisis_endpoints, endpoints_lentos, errores, logs_parseados)

    return {
        'stats_general': stats_general,
        'analisis_codigos': analisis_codigos,
        'analisis_endpoints': analisis_endpoints,
        'endpoints_lentos': endpoints_lentos,
        'errores': errores
    }

def generar_reporte_plantilla(stats_general, analisis_codigos, analisis_endpoints, endpoints_lentos, errores, logs_parseados):

    with open('plantilla_reporte.md', 'r', encoding='utf-8') as f:
        plantilla = f.read()

    plantilla = plantilla.replace('[FECHA]', datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S'))
    plantilla = plantilla.replace('[TOTAL_LINEAS]', str(len(logs_parseados)))
    plantilla = plantilla.replace('[INICIO]', logs_parseados[0]['timestamp'] if logs_parseados else 'N/A')
    plantilla = plantilla.replace('[FIN]', logs_parseados[-1]['timestamp'] if logs_parseados else 'N/A')

    plantilla = plantilla.replace('[TOTAL]', str(stats_general['total']))
    plantilla = plantilla.replace('[PROMEDIO]', f"{stats_general['promedio']:.1f}")
    plantilla = plantilla.replace('[MAX]', f"{stats_general['maximo']:.1f}")
    plantilla = plantilla.replace('[MIN]', f"{stats_general['minimo']:.1f}")
    plantilla = plantilla.replace('[DESVIACION]', f"{stats_general['desviacion']:.1f}")

    codigos_especificos = {200: '[N]', 201: '[N]', 404: '[N]', 500: '[N]'}
    otros_cantidad = 0
    otros_tiempo = 0

    for codigo, datos in analisis_codigos.items():
        if codigo in codigos_especificos:
            placeholder_n = f"[N]"
            placeholder_porcentaje = f"[%]"
            placeholder_time = f"[TIME]"

            plantilla = plantilla.replace(f"{codigo}    | [N]      | [%]        | [TIME] ms",
                                        f"{codigo}    | {datos['cantidad']}      | {datos['porcentaje']:.1f}%        | {datos['tiempo_promedio']:.1f} ms")
        else:
            otros_cantidad += datos['cantidad']
            otros_tiempo += datos['tiempo_promedio'] * datos['cantidad']

    tiempo_promedio_otros = otros_tiempo / otros_cantidad if otros_cantidad > 0 else 0
    porcentaje_otros = (otros_cantidad / stats_general['total']) * 100

    plantilla = plantilla.replace("Otros  | [N]      | [%]        | [TIME] ms",
                                f"Otros  | {otros_cantidad}      | {porcentaje_otros:.1f}%        | {tiempo_promedio_otros:.1f} ms")

    endpoints_especificos = ['/api/users', '/api/products', '/api/login']

    for endpoint in endpoints_especificos:
        if endpoint in analisis_endpoints:
            datos = analisis_endpoints[endpoint]
            plantilla = plantilla.replace(
                f"{endpoint} | [N] | [AVG] ms | [MAX] ms | [MIN] ms",
                f"{endpoint} | {datos['solicitudes']} | {datos['tiempo_promedio']:.1f} ms | {datos['tiempo_maximo']:.1f} ms | {datos['tiempo_minimo']:.1f} ms"
            )

    for i in range(3):
        if i < len(endpoints_lentos):
            endpoint, datos = endpoints_lentos[i]
            plantilla = plantilla.replace(f"[ENDPOINT_{i+1}]", endpoint)
            plantilla = plantilla.replace(f"[TIME]", f"{datos['tiempo_promedio']:.1f}")
            plantilla = plantilla.replace(f"[N]", str(datos['solicitudes']))
            plantilla = plantilla.replace(f"[MAX]", f"{datos['tiempo_maximo']:.1f}")

    if errores:
        endpoints_errores = {}
        codigos_errores = {}
        for error in errores:
            endpoint = error['endpoint']
            codigo = error['codigo_estado']
            endpoints_errores[endpoint] = endpoints_errores.get(endpoint, 0) + 1
            codigos_errores[codigo] = codigos_errores.get(codigo, 0) + 1

        endpoint_mas_errores = max(endpoints_errores.items(), key=lambda x: x[1])
        error_mas_comun = max(codigos_errores.items(), key=lambda x: x[1])

        plantilla = plantilla.replace("[TOTAL_ERRORES]", str(len(errores)))
        plantilla = plantilla.replace("[PORCENTAJE]", f"{(len(errores) / stats_general['total']) * 100:.1f}")
        plantilla = plantilla.replace("[ENDPOINT]", f"{endpoint_mas_errores[0]} ({endpoint_mas_errores[1]} errores)")
        plantilla = plantilla.replace("[CODIGO]", str(error_mas_comun[0]))
        plantilla = plantilla.replace("[CANTIDAD]", str(error_mas_comun[1]))

    patrones = []

    for endpoint, datos in analisis_endpoints.items():
        if datos['tiempo_promedio'] > 500:
            patrones.append(f"Se observa un incremento en los tiempos de respuesta para el endpoint {endpoint}")

    if errores:
        for endpoint, count in endpoints_errores.items():
            if count > 1:
                patrones.append(f"Los errores {list(codigos_errores.keys())} estan concentrados en el endpoint {endpoint}")
                break
        else:
            primer_error = errores[0]
            patrones.append(f"Los errores estan distribuidos, el mas frecuente es {error_mas_comun[0]} en {endpoint_mas_errores[0]}")

    patrones_texto = "\n".join(f"- {patron}" for patron in patrones) if patrones else "- No se identificaron patrones significativos"
    plantilla = plantilla.replace("[DESCRIBIR PATRONES OBSERVADOS]", patrones_texto)

    # Reemplazar recomendaciones
    recomendaciones = []

    if endpoints_lentos:
        endpoint, datos = endpoints_lentos[0]
        recomendaciones.append({
            'descripcion': f"Optimizar el endpoint {endpoint}",
            'impacto': "Reduccion significativa en el tiempo de respuesta general del sistema",
            'prioridad': "Alta"
        })

    if len(errores) > 0:
        recomendaciones.append({
            'descripcion': "Revisar y corregir endpoints con errores HTTP",
            'impacto': "Mejora en la confiabilidad y experiencia del usuario",
            'prioridad': "Media"
        })

    if stats_general['desviacion'] > 300:
        recomendaciones.append({
            'descripcion': "Estabilizar la variabilidad en los tiempos de respuesta",
            'impacto': "Mayor consistencia en el rendimiento del sistema",
            'prioridad': "Media"
        })

    while len(recomendaciones) < 3:
        recomendaciones.append({
            'descripcion': "Monitoreo continuo del rendimiento",
            'impacto': "Deteccion temprana de problemas de rendimiento",
            'prioridad': "Baja"
        })

    for i in range(3):
        if i < len(recomendaciones):
            rec = recomendaciones[i]
            plantilla = plantilla.replace(f"[RECOMENDACION_{i+1}]", rec['descripcion'])
            plantilla = plantilla.replace(f"[DESCRIPCION]", rec['impacto'])
            plantilla = plantilla.replace(f"[Alta/Media/Baja]", rec['prioridad'])

    conclusiones = f"El analisis de rendimiento realizado sobre {stats_general['total']} solicitudes revela que el sistema presenta un tiempo de respuesta promedio de {stats_general['promedio']:.1f} ms. Se identificaron {len(errores)} solicitudes con errores que requieren atencion. Los endpoints mas criticos han sido priorizados para optimizacion."
    conclusiones = f"Este taller demostró la aplicación práctica de los conceptos de diseño funcional en Python, donde mediante el uso de subalgoritmos, funciones puras, closures, lambdas y funciones de orden superior como filter, map y sorted, se logró construir un sistema completo de análisis de rendimiento capaz de procesar logs reales, calcular métricas estadísticas y generar reportes automatizados, evidenciando cómo la programación funcional permite crear código modular, mantenible y eficiente para el procesamiento de datos. Adicionalmente, el análisis de rendimiento realizado sobre {stats_general['total']} solicitudes revela que el sistema presenta un tiempo de respuesta promedio de {stats_general['promedio']:.1f} ms, con {len(errores)} solicitudes con errores que requieren atención y endpoints críticos priorizados para optimización."
    plantilla = plantilla.replace("[TUS CONCLUSIONES SOBRE EL ANALISIS]", conclusiones)

    with open('reporte_rendimiento.md', 'w', encoding='utf-8') as f:
        f.write(plantilla)

    print(f"Reporte generado: reporte_rendimiento.md")
    print(f"- {stats_general['total']} solicitudes analizadas")
    print(f"- {len(analisis_codigos)} codigos de estado diferentes")
    print(f"- {len(analisis_endpoints)} endpoints unicos")
    print(f"- {len(errores)} errores identificados")

print("Ejecutando analisis completo...")
resultados = main()

Ejecutando analisis completo...
INICIANDO ANALISIS COMPLETO DE LOGS
50 logs procesados correctamente
Reporte generado: reporte_rendimiento.md
- 50 solicitudes analizadas
- 5 codigos de estado diferentes
- 44 endpoints unicos
- 3 errores identificados
